In [33]:
import tensorflow as tf
from tensorflow.keras.layers import Dense
print("TensorFlow version:", tf.__version__)
# Load the TensorBoard notebook extension
%load_ext tensorboard
import datetime
import shutil

TensorFlow version: 2.18.0
The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


In [34]:
%reload_ext tensorboard
from tensorflow.keras import Model
from pathlib import Path
import pandas as pd

In [35]:
BATCH_SIZE = 128   
BUFFER_SIZE = 512   
LEARNING_RATE = 0.00001
EPOCHS = 6000 
# best results 

In [36]:
input_dir = Path('./prepared')
logs_path = Path('./logs')
if logs_path.exists():
  shutil.rmtree(logs_path) # удаляем, если существует /logs
logs_path.mkdir(parents=True)

X_train_name = input_dir / 'X_train.csv'
y_train_name = input_dir / 'y_train.csv'
X_test_name = input_dir / 'X_test.csv'
y_test_name = input_dir / 'y_test.csv'

X_train = pd.read_csv(X_train_name)
y_train = pd.read_csv(y_train_name)
X_test = pd.read_csv(X_test_name)
y_test = pd.read_csv(y_test_name)

train_ds = tf.data.Dataset.from_tensor_slices(
    (X_train, y_train)).shuffle(BUFFER_SIZE).batch(BATCH_SIZE)

test_ds = tf.data.Dataset.from_tensor_slices((X_test, y_test)).batch(BATCH_SIZE)

In [37]:
@tf.keras.utils.register_keras_serializable() #  Декоратор позволяет сериализовать и десериализовать модель для сохранения и загрузки.
class SomeModel(Model):
    def __init__(self, neurons_cnt=64, **kwargs):
        super(SomeModel, self).__init__(**kwargs)
        self.neurons_cnt = neurons_cnt  # Сохраняем значение параметра для конфигурации
        self.d_in = Dense(30, activation='relu')
        self.d1 = Dense(neurons_cnt, activation='relu')
        self.d_out = Dense(1)

    def call(self, x):
        x = self.d_in(x)
        x = self.d1(x)
        return self.d_out(x)
         
    def build(self, input_shape): # надо явно определить для построения
        super(SomeModel, self).build(input_shape)
        
    def get_config(self): 
        # Возвращаем параметры модели, включая кастомные
        config = super(SomeModel, self).get_config()
        config.update({
            "neurons_cnt": self.neurons_cnt  # Добавляем кастомный параметр в конфигурацию
        })
        return config

    @classmethod
    def from_config(cls, config):
        # Создаём экземпляр класса из конфигурации
        return cls(**config)

In [38]:
# Create an instance of the model
model = SomeModel(neurons_cnt=32)
model.build(input_shape=(None, 30))

In [39]:
loss_object = tf.keras.losses.MeanSquaredError() 
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)

train_loss = tf.keras.metrics.Mean(name='train_loss')
train_accuracy = tf.keras.metrics.MeanAbsoluteError(name='train_mae')

test_loss = tf.keras.metrics.Mean(name='test_loss')
test_accuracy = tf.keras.metrics.MeanAbsoluteError(name='test_mae')

In [40]:
@tf.function
def train_step(input_vector, labels):
  with tf.GradientTape() as tape:
    # training=True is only needed if there are layers with different
    # behavior during training versus inference (e.g. Dropout).
    predictions = model(input_vector, training=True)
    loss = loss_object(labels, predictions)
  gradients = tape.gradient(loss, model.trainable_variables)
  optimizer.apply_gradients(zip(gradients, model.trainable_variables))

  train_loss(loss)
  train_accuracy(labels, predictions)

@tf.function
def test_step(input_vector, labels):
  # training=False is only needed if there are layers with different
  # behavior during training versus inference (e.g. Dropout).
  predictions = model(input_vector, training=False)
  t_loss = loss_object(labels, predictions)

  test_loss(t_loss)
  test_accuracy(labels, predictions)

In [41]:
from tensorflow import keras
current_time = datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
train_log_dir = logs_path / 'gradient_tape' / current_time / 'train'
train_log_dir.mkdir(exist_ok=True, parents=True)
test_log_dir = logs_path / 'gradient_tape' / current_time / 'test'
test_log_dir.mkdir(exist_ok=True, parents=True)
train_summary_writer = tf.summary.create_file_writer(str(train_log_dir))
test_summary_writer = tf.summary.create_file_writer(str(test_log_dir))

logdir = logs_path / "fit" / datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
logdir.mkdir(exist_ok=True, parents=True)
fit_summary_writer = tf.summary.create_file_writer(str(logdir))

tf.summary.trace_on(graph=True, profiler=True, profiler_outdir=str(logdir))

for epoch in range(EPOCHS):
  # Reset the metrics at the start of the next epoch
  for (x_train, y_train) in train_ds:

    with fit_summary_writer.as_default():
      train_step(x_train, y_train)


  with train_summary_writer.as_default():
    tf.summary.scalar('loss', train_loss.result(), step=epoch)
    tf.summary.scalar('accuracy', train_accuracy.result(), step=epoch)

  for (x_test, y_test) in test_ds:
    test_step(x_test, y_test)

  with test_summary_writer.as_default():
    tf.summary.scalar('loss', test_loss.result(), step=epoch)
    tf.summary.scalar('mae', test_accuracy.result(), step=epoch)
    for layer in model.layers:
        for weight in layer.weights:
            tf.summary.histogram(f"{layer.name}/{weight.name}", weight, step=epoch)
            
  template = 'Epoch {}, Loss: {}, Accuracy: {}, Test Loss: {}, Test MAE: {}'
  print (template.format(epoch+1,
                         train_loss.result(),
                         train_accuracy.result(),
                         test_loss.result(),
                         test_accuracy.result()))

  # Reset metrics every epoch
  train_loss.reset_state()
  test_loss.reset_state()
  train_accuracy.reset_state()
  test_accuracy.reset_state()

with fit_summary_writer.as_default():
  tf.summary.trace_export(
      name="my_func_trace",
      step=0,
      profiler_outdir=str(logdir)
  )

Epoch 1, Loss: 206.56394958496094, Accuracy: 14.361007690429688, Test Loss: 203.6022186279297, Test MAE: 14.243008613586426
Epoch 2, Loss: 199.14610290527344, Accuracy: 14.103864669799805, Test Loss: 196.3418426513672, Test MAE: 13.985160827636719
Epoch 3, Loss: 192.16751098632812, Accuracy: 13.844585418701172, Test Loss: 189.1272430419922, Test MAE: 13.724053382873535
Epoch 4, Loss: 185.00076293945312, Accuracy: 13.580779075622559, Test Loss: 181.89894104003906, Test MAE: 13.457266807556152
Epoch 5, Loss: 178.07864379882812, Accuracy: 13.311685562133789, Test Loss: 174.67132568359375, Test MAE: 13.185060501098633
Epoch 6, Loss: 170.79981994628906, Accuracy: 13.035945892333984, Test Loss: 167.4213104248047, Test MAE: 12.906158447265625
Epoch 7, Loss: 163.62109375, Accuracy: 12.754780769348145, Test Loss: 160.20855712890625, Test MAE: 12.622450828552246
Epoch 8, Loss: 156.10850524902344, Accuracy: 12.468345642089844, Test Loss: 153.0135498046875, Test MAE: 12.332836151123047
Epoch 9, Lo

In [42]:
%tensorboard --logdir ./logs/gradient_tape --port=8353

Reusing TensorBoard on port 8353 (pid 19448), started 0:48:06 ago. (Use '!kill 19448' to kill it.)

In [44]:
%load_ext tensorboard
%tensorboard --logdir=./logs/fit --port=8665

The tensorboard extension is already loaded. To reload it, use:
  %reload_ext tensorboard


Reusing TensorBoard on port 8665 (pid 20736), started 0:53:33 ago. (Use '!kill 20736' to kill it.)

In [45]:
tensorboard --logdir logs/weights/

Reusing TensorBoard on port 6006 (pid 16516), started 1 day, 4:16:35 ago. (Use '!kill 16516' to kill it.)